In [ ]:
!pip install -q lightgbm xgboost mlflow pyarrow

In [ ]:
import json
import os
from pathlib import Path

import lightgbm as lgb
import mlflow
import numpy as np
import pandas as pd
import xgboost as xgb
from kaggle_secrets import UserSecretsClient


def find_input(filename, root="/kaggle/input"):
    matches = sorted(Path(root).rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"expected exactly one {filename} under {root}, found {len(matches)}: {matches}"
        )
    return matches[0]


features_df = pd.read_parquet(find_input("demandlens_features.parquet"))
features_df["date"] = pd.to_datetime(features_df["date"])
features_df = features_df.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)

secrets = UserSecretsClient()
DAGSHUB_TOKEN = secrets.get_secret("DAGSHUB_TOKEN")
DAGSHUB_REPO = secrets.get_secret("DAGSHUB_REPO")

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_TOKEN
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_REPO}.mlflow")
mlflow.set_experiment("demandlens_ml_models")

In [ ]:
HORIZON = 15
N_FOLDS = 4


def get_walkforward_folds(all_dates, horizon=HORIZON, n_folds=N_FOLDS):
    all_dates = pd.DatetimeIndex(all_dates).sort_values()
    last_date = all_dates[-1]
    holdout_start = last_date - pd.Timedelta(days=horizon - 1)

    folds = []
    for i in range(n_folds, 0, -1):
        test_end = holdout_start - pd.Timedelta(days=(i - 1) * horizon + 1)
        test_start = test_end - pd.Timedelta(days=horizon - 1)
        train_end = test_start - pd.Timedelta(days=1)
        folds.append((train_end, test_start, test_end))

    holdout = (holdout_start - pd.Timedelta(days=1), holdout_start, last_date)
    return folds, holdout


all_dates = features_df["date"].unique()
folds, holdout = get_walkforward_folds(all_dates)
print("CV folds (train_end, test_start, test_end):")
for f in folds:
    print(f)
print("Holdout:", holdout)

In [ ]:
import numpy as np


def mape(actual, forecast):
    actual, forecast = np.array(actual), np.array(forecast)
    mask = actual != 0
    if not mask.any():
        return np.nan
    return np.mean(np.abs((actual[mask] - forecast[mask]) / actual[mask])) * 100


def wape(actual, forecast):
    actual, forecast = np.array(actual), np.array(forecast)
    denom = np.sum(np.abs(actual))
    if denom == 0:
        return np.nan
    return np.sum(np.abs(actual - forecast)) / denom * 100


def mase(actual, forecast, train_series, seasonal_period=7):
    actual, forecast = np.array(actual), np.array(forecast)
    train_series = np.array(train_series)
    naive_errors = np.abs(train_series[seasonal_period:] - train_series[:-seasonal_period])
    if len(naive_errors) == 0:
        return np.nan
    scale = np.mean(naive_errors)
    if scale == 0:
        return np.nan
    return np.mean(np.abs(actual - forecast)) / scale

In [ ]:
FEATURE_COLS = [
    "store_nbr", "family", "city", "state", "type", "cluster",
    "onpromotion", "is_holiday", "day_of_week", "month", "is_weekend",
    "day_of_month", "week_of_year", "dcoilwtico_lag",
    "lag_21", "lag_28", "lag_35",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_28",
    "rolling_std_7", "rolling_std_14", "rolling_std_28",
]
CATEGORICAL_COLS = ["store_nbr", "family", "city", "state", "type", "cluster"]
TARGET = "sales"
SERIES_KEYS = ["store_nbr", "family"]

for col in CATEGORICAL_COLS:
    features_df[col] = features_df[col].astype("category")


def run_fold(model_type, train_df, test_df):
    X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET]
    X_test = test_df[FEATURE_COLS]

    if model_type == "lightgbm":
        model = lgb.LGBMRegressor(
            n_estimators=500, learning_rate=0.05, num_leaves=31, verbose=-1,
        )
        model.fit(X_train, y_train, categorical_feature=CATEGORICAL_COLS)
    elif model_type == "xgboost":
        model = xgb.XGBRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            enable_categorical=True, tree_method="hist",
        )
        model.fit(X_train, y_train)
    else:
        raise ValueError(f"Unknown model_type: {model_type!r}")

    pred = np.clip(model.predict(X_test), 0, None)
    return model, pred


def per_series_metrics(train_df, test_df, pred):
    history = {
        key: group.sort_values("date")[TARGET].to_numpy()
        for key, group in train_df.groupby(SERIES_KEYS, observed=True)
    }
    scored = test_df[SERIES_KEYS + [TARGET]].assign(pred=np.asarray(pred))
    rows = []
    for key, group in scored.groupby(SERIES_KEYS, observed=True):
        actual = group[TARGET].to_numpy()
        forecast = group["pred"].to_numpy()
        rows.append({
            "store_nbr": int(key[0]), "family": str(key[1]),
            "mape": mape(actual, forecast),
            "wape": wape(actual, forecast),
            "mase": mase(actual, forecast, history[key]),
        })
    return pd.DataFrame(rows)

In [ ]:
ml_results = []
final_holdout_preds = {}

for model_type in ["lightgbm", "xgboost"]:
    with mlflow.start_run(run_name=f"{model_type}_global_walkforward"):
        mlflow.log_param("model_type", model_type)
        mlflow.log_param("horizon", HORIZON)
        mlflow.log_param("n_folds", N_FOLDS)

        for fold_idx, (train_end, test_start, test_end) in enumerate(folds + [holdout]):
            fold_label = f"fold_{fold_idx+1}" if fold_idx < len(folds) else "holdout"

            train_df = features_df[features_df["date"] <= train_end]
            test_df = features_df[
                (features_df["date"] >= test_start) & (features_df["date"] <= test_end)
            ]
            if len(train_df) == 0 or len(test_df) == 0:
                continue

            model, pred = run_fold(model_type, train_df, test_df)

            fold_metrics = per_series_metrics(train_df, test_df, pred)
            fold_metrics.insert(0, "fold", fold_label)
            fold_metrics.insert(0, "model", model_type)
            ml_results.append(fold_metrics)

            for metric in ["mape", "wape", "mase"]:
                mlflow.log_metric(f"{fold_label}_{metric}", float(fold_metrics[metric].mean()))

            if fold_label == "holdout":
                test_df_result = test_df[["date", "store_nbr", "family", TARGET]].copy()
                test_df_result["store_nbr"] = test_df_result["store_nbr"].astype(int)
                test_df_result["family"] = test_df_result["family"].astype(str)
                test_df_result["forecast"] = pred
                final_holdout_preds[model_type] = test_df_result

ml_results_df = pd.concat(ml_results, ignore_index=True)
print(ml_results_df.groupby(["model", "fold"])[["mape", "wape", "mase"]].mean())

In [ ]:
FAMILY_COST_PER_UNIT_ERROR = {
    "GROCERY I": 0.75, "BEVERAGES": 0.75, "CLEANING": 0.75, "HOME CARE": 0.75,
    "PRODUCE": 0.50, "DAIRY": 0.50,
}

cost_summary = {}
for model_type, df in final_holdout_preds.items():
    df = df.copy()
    df["cost_per_unit"] = df["family"].map(FAMILY_COST_PER_UNIT_ERROR)
    df["abs_error"] = np.abs(df["sales"] - df["forecast"])
    estimated_cost = (df["abs_error"] * df["cost_per_unit"]).sum()
    total_abs_error = df["abs_error"].sum()
    cost_summary[model_type] = {
        "total_abs_error_units": float(total_abs_error),
        "estimated_cost_usd": float(estimated_cost),
    }
    print(f"{model_type}: total abs error units = {total_abs_error:.0f}, estimated cost = ${estimated_cost:,.2f}")

with open("/kaggle/working/cost_of_error.json", "w", encoding="utf-8") as f:
    json.dump(cost_summary, f, indent=2)
print("Saved: cost_of_error.json")

In [ ]:
ml_results_df.to_csv("/kaggle/working/ml_results.csv", index=False)

cv_mase = (
    ml_results_df[ml_results_df["fold"] != "holdout"]
    .groupby("model")["mase"].mean()
    .sort_values()
)
print("Mean MASE over CV folds:")
print(cv_mase)
best_model = cv_mase.index[0]
print(f"Selected model (lowest CV MASE, holdout not used): {best_model}")

final_holdout_output = final_holdout_preds[best_model].copy()
final_holdout_output["model"] = best_model
final_holdout_output.to_parquet("/kaggle/working/final_holdout_predictions.parquet", index=False)
print(f"Saved: ml_results.csv, final_holdout_predictions.parquet, cost_of_error.json (using {best_model} predictions)")